# ITSPM: Colab quickstart
Choose **Runtime → Change runtime type → GPU**, then run the setup and small examples.
These examples check the pipeline; they do not reproduce full benchmark scores.
The default data are bundled or generated locally. P12, PAM, USHCN, Stocks and Energy need separate preparation; see the [data guide](https://github.com/zhouziyu02/IP_ITSPM/blob/main/docs/DATASETS.md).


In [1]:
%cd /content
!git clone https://github.com/zhouziyu02/IP_ITSPM.git
%cd /content/IP_ITSPM
%pip install -q -r requirements.txt


/content
Cloning into 'IP_ITSPM'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 133 (delta 11), reused 133 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 20.70 MiB | 28.34 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/IP_ITSPM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.7 MB/s eta 0:00:00


In [2]:
import torch
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!python3 scripts/check_data.py --dataset bundled --verify


PyTorch: 2.11.0+cu130 CUDA available: True
GPU: Tesla T4
OK activity
OK taxi
OK mujoco
VERIFIED data/activity/processed/data.pt
VERIFIED TPP/taxi/train.pkl
VERIFIED TPP/taxi/dev.pkl
VERIFIED TPP/taxi/test.pkl
VERIFIED Generation/itspm_generation/table1_data/mujoco_training_36.pt


## 1. Small real-data forecasting run
Activity is already bundled. This uses eight source records, one epoch and a small model.
Training/validation/test identities are split before window expansion. The full preset is `Extrapolation/scripts/activity.sh`.


In [3]:
!python3 regression.py --model ITSPM --dataset activity --task forecasting --history 3000 -n 8 --epoch 1 --batch_size 8 --d_model 16 --n_ref_points 8 --n_scales 2 --n_mixer_layers 1 --max_event_tokens 16 --max_gap_tokens 8 --seed 0 --state colab_quickstart


PID, device: 775 cuda
/content/IP_ITSPM/regression.py
2026-10-23 06:53:58
regression.py --model ITSPM --dataset activity --task forecasting --history 3000 -n 8 --epoch 1 --batch_size 8 --d_model 16 --n_ref_points 8 --n_scales 2 --n_mixer_layers 1 --max_event_tokens 16 --max_gap_tokens 8 --seed 0 --state colab_quickstart
Namespace(state='debug', model='ITSPM', root_path='', data_path='/content/IP_ITSPM/data', n=8, epoch=1, batch_size=8, num_types=23, d_model=16, dropout=0.0, lr=0.001, gpu='0', log='./logs/', save_path='./checkpoints/regression/', seed=0, patience=10, weight_decay=1e-05, load_path=None, test_only=False, logmode='a', task='forecasting', debug_flag=False, dp_flag=False, load_in_batch=False, history=3000, retrain=False, median_len=50, load=None, dataset='activity', quantization=0.0, max_len=-1, sample_rate=1.0, mask_rate=0.3, collate='indseq', n_ref_points=8, n_scales=2, n_mixer_layers=1, max_event_tokens=16, max_gap_tokens=8, device=device(type='cuda'), PID=775)
Total reco

## 2. Small generation run
Sines needs no external download. Two epochs and ten discriminator steps are only a functionality check, not a useful quality estimate.


In [4]:
!SEQ_LENS=12 MISSING_RATIOS=0.5 N_SAMPLES=256 DS_ITERATIONS=10 GPU=0 SEED=0 bash Generation/scripts/sines.sh --ae_epochs 2 --diff_epochs 2 --batch_size 16


/content/IP_ITSPM/models/ITSPM.py:512: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.token_mixer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
ae_epoch=001 loss=0.938494
diff_epoch=001 loss=1.000238
{
  "gaussian_baseline": {
    "acf_error": 0.010900115594267845,
    "ds": 0.0,
    "kl": 0.03789685250212398,
    "mdd": 3.9755685329437256,
    "mmd": 0.035642750561674985
  },
  "itspm_latent_diffusion": {
    "acf_error": 0.01912151649594307,
    "ds": 0.0,
    "kl": 9.46407224726602,
    "mdd": 6.843851566314697,
    "mmd": 0.31691921774590215
  }
}


## 3. CPU regression checks
This covers all task families, including classification on synthetic fixtures and a real Taxi batch.


In [5]:
!bash scripts/run_checks.sh


OK activity
OK taxi
OK mujoco
VERIFIED data/activity/processed/data.pt
VERIFIED TPP/taxi/train.pkl
VERIFIED TPP/taxi/dev.pkl
VERIFIED TPP/taxi/test.pkl
VERIFIED Generation/itspm_generation/table1_data/mujoco_training_36.pt
test_both_classification_collates (test_core.CoreTests.test_both_classification_collates) ... ok
test_checkpoint_roundtrip (test_core.CoreTests.test_checkpoint_roundtrip) ... ok
ok
test_empty_windows_and_masks (test_core.CoreTests.test_empty_windows_and_masks) ... Excluded 1 records/windows without observed inputs or prediction targets.
Excluded 1 records/windows without observed inputs or prediction targets.
Excluded 1 USHCN windows without observed inputs or prediction targets.
ok
test_error_metrics (test_core.CoreTests.test_error_metrics) ... ok
test_invalid_input_shapes_are_rejected (test_core.CoreTests.test_invalid_input_shapes_are_rejected) ... ok
test_model_forward_and_backward (test_core.CoreTests.test_model_forward_and_backward) ... ok
test_partial_class_met

## Continue with your project
Read [README](https://github.com/zhouziyu02/IP_ITSPM), [project milestones](https://github.com/zhouziyu02/IP_ITSPM/blob/main/docs/PROJECTS.md), and [reference-result limitations](https://github.com/zhouziyu02/IP_ITSPM/blob/main/docs/REFERENCE_RESULTS.md).

After preparing the chosen dataset, run its preset with explicit seeds 0, 1 and 2, keep splits and budgets fixed, and report mean ± sample standard deviation. Trailing flags such as `--epoch 5 --batch_size 8` override classification/regression presets. The generation head still needs a predictive-score evaluator for project 1.5; projects 1.6–1.8 require extensions.

Checkpoints, results and logs are local and ignored by Git. Copy them to your own persistent storage before the Colab runtime disconnects; mounted Google Drive is one option. Do not treat a short run or historical single-seed result as a completed reproduction.
